# General Relativity Module

This module provides the foundational tools and numerical infrastructure needed to work with General Relativity (GR) in a modern, performant, and extensible way. It is designed to support both symbolic and numerical computations related to curved spacetimes, tensorial quantities, and dynamical evolutions in a 4D or 3+1 formulation.

---

## Objectives

- Represent tensors, metrics, and connections in arbitrary coordinate systems.
- Support symbolic parsing of Einstein field equations and related expressions.
- Provide efficient numerical backends for computing curvature (Christoffel symbols, Riemann, Ricci, Einstein tensors).
- Support BSSN formalism and numerical relativity applications.

---


## Key Features

- High-performance tensor algebra using SIMD and OpenMP.
- Built-in support for Kerr, Schwarzschild, Minkowski, and user-defined metrics.
- Futur integration with numerical evolution schemes (RK4, flux solvers).

---

## When to Use This Module

This module is intended for developers, researchers, and students working on:

- Black hole simulations
- Cosmological modeling
- Numerical relativity
- High-performance scientific computing


### Schwarzschild Metric and Its Inverse

In this example, we demonstrate how to define and use a custom metric in Morpheus.  
(The Schwarzschild metric is already available via the built-in `morpheus_RG::Metric<double> metric("Schwarzschild", 1.0, 0.0);` constructor, where the two arguments are mass and spin.)

In general relativity, the Schwarzschild metric describes the curvature of spacetime around a static, spherically symmetric mass.  
Its line element is given by:

$$
ds^2 = -\left(1 - \frac{2M}{r} \right) dt^2 + \left(1 - \frac{2M}{r} \right)^{-1} dr^2 + r^2 d\theta^2 + r^2 \sin^2 \theta \, d\phi^2
$$

This corresponds to a diagonal $4 \times 4$ metric tensor $g_{\mu\nu}$ in spherical coordinates $(t, r, \theta, \phi)$:

$$
g_{\mu\nu} = \begin{pmatrix}
- f(r) & 0 & 0 & 0 \\
0 & f(r)^{-1} & 0 & 0 \\
0 & 0 & r^2 & 0 \\
0 & 0 & 0 & r^2 \sin^2 \theta
\end{pmatrix}, \quad \text{with } f(r) = 1 - \frac{2M}{r}
$$

Although this matrix is diagonal and its inverse is analytically trivial, we compute the inverse numerically using a general-purpose matrix inversion routine.  
This serves both as a validation of the numerical backend and a practical example for working with symbolic or nontrivial metrics.

This example is a foundational step toward computing Christoffel symbols, the Ricci tensor, and full curvature tensors in numerical relativity simulations.

In [7]:
import sys
import time 
import os
sys.path.append(os.path.abspath("../pybuild"))
from morpheus import *
from morpheus import Matrix, morph
import math

def schwarzschild_metric(r, theta, M=1.0):
    g = Matrix(4, 4)
    f = 1.0 - 2.0 * M / r

    g[0, 0] = -f
    g[1, 1] = 1.0 / f
    g[2, 2] = r**2
    g[3, 3] = r**2 * math.sin(theta)**2

    return g

r = 10.0
theta = math.pi / 4
g = schwarzschild_metric(r, theta)

g_inv = morph.inverse_mat(g)

print("Metric g_mu_nu at (r, θ):")
g.print()
print("Inverse metric g^mu_nu:")
g_inv.print()


Metric g_mu_nu at (r, θ):
Inverse metric g^mu_nu:
[ -0.800000 0.000000 0.000000 0.000000 ]
[ 0.000000 1.250000 0.000000 0.000000 ]
[ 0.000000 0.000000 100.000000 0.000000 ]
[ 0.000000 0.000000 0.000000 50.000000 ]
[ -1.250000 0.000000 0.000000 0.000000 ]
[ 0.000000 0.800000 0.000000 0.000000 ]
[ 0.000000 0.000000 0.010000 0.000000 ]
[ 0.000000 0.000000 0.000000 0.020000 ]


### Kerr Geometry: Computing the Riemann Curvature Tensor

In this example, we compute the **Riemann curvature tensor** $R^\lambda_{\ \mu\nu\rho}$ at a specific spacetime point in the Kerr metric, a solution to Einstein's field equations representing a rotating black hole.

We evaluate at the following point in Boyer–Lindquist coordinates:

$$
X^\mu = (t, r, \theta, \phi) = \left(0,\ 10,\ \frac{\pi}{2},\ 0\right)
$$

with Kerr parameters:
- $M = 1.0$ (mass of the black hole)
- $a = 0.8$ (angular momentum per unit mass)

---

#### 1. Metric Tensor

We construct the Kerr metric $g_{\mu\nu}(X)$ at this point with:

```python
g = morph.Metric("kerr", 1.0, 0.8)(X)
```

This returns a $4 \times 4$ symmetric matrix representing the spacetime geometry.  
We compute its inverse $g^{\mu\nu}$ via:

```python
g_inv = morph.inv_mat_tensor(g)
```

---

#### 2. Christoffel Symbols

The Christoffel symbols $\Gamma^\lambda_{\mu\nu}$ describe how vectors are transported in curved space and are computed as:

```python
Gamma = morph.compute_christoffel(X, g, g_inv, "kerr", 1.0, 0.8)
```

Their definition is:

$$
\Gamma^\lambda_{\mu\nu} = \frac{1}{2} g^{\lambda\sigma} \left( \partial_\mu g_{\nu\sigma} + \partial_\nu g_{\mu\sigma} - \partial_\sigma g_{\mu\nu} \right)
$$

Finite difference approximations are used for the partial derivatives.

---

#### 3. Riemann Tensor

The Riemann tensor is computed from the Christoffel symbols using:

```python
R = morph.compute_riemann_tensor(X, "kerr", 1.0, 0.8)
```

The formula is:

$$
R^\lambda_{\ \mu\nu\rho} = \partial_\nu \Gamma^\lambda_{\mu\rho}
- \partial_\rho \Gamma^\lambda_{\mu\nu}
+ \Gamma^\lambda_{\nu\sigma} \Gamma^\sigma_{\mu\rho}
- \Gamma^\lambda_{\rho\sigma} \Gamma^\sigma_{\mu\nu}
$$

It encodes the tidal and curvature effects of spacetime.

We print the components sliced by upper index $\lambda$:

```python
morph.Riemann.print_componentwise(R)
```

---

### Summary

This pipeline performs a **numerical differential geometry evaluation** in general relativity:
- Builds the Kerr metric analytically
- Computes the Christoffel symbols $\Gamma^\lambda_{\mu\nu}$ numerically
- Computes the Riemann tensor $R^\lambda_{\ \mu\nu\rho}$
- Displays the full curvature structure at the point $X^\mu$

This reveals how spacetime is curved near a rotating black hole, illustrating gravitational tidal forces and frame-dragging effects.


In [3]:
from morpheus import Vectord, morph

X = Vectord([0.0, 10.0, 3.14159 / 2, 0.0])
g = morph.Metric("kerr", 1.0, 0.935)(X)
g_inv = morph.inv_mat_tensor(g)
Gamma = morph.compute_christoffel(X, g, g_inv, "kerr", 1.0, 0.935)
Gamma.print()
Riemann = morph.compute_riemann_tensor(X, "kerr", 1.0, 0.935)
Ricci = morph.contract_riemann_to_ricci(Riemann, g_inv)
morph.Riemann.print_componentwise(Riemann)
morph.print_ricci_tensor(Ricci)
Rscalar = morph.print_ricci_scalar(Ricci, g_inv)



Γ^0_{μν} :
    0.000000     0.012473    -0.000000     0.000000 
    0.012473     0.000000     0.000000    -0.034785 
   -0.000000     0.000000     0.000000     0.000000 
    0.000000    -0.034785     0.000000     0.000000 

Γ^1_{μν} :
    0.008087     0.000000     0.000000    -0.007562 
    0.000000    -0.011284    -0.000000     0.000000 
    0.000000    -0.000000    -8.087422     0.000000 
   -0.007562     0.000000     0.000000    -8.080352 

Γ^2_{μν} :
   -0.000000     0.000000     0.000000     0.000000 
    0.000000     0.000000     0.100000     0.000000 
    0.000000     0.100000    -0.000000     0.000000 
    0.000000     0.000000     0.000000    -0.000001 

Γ^3_{μν} :
    0.000000     0.000116    -0.000000     0.000000 
    0.000116     0.000000     0.000000     0.098811 
   -0.000000     0.000000     0.000000     0.000001 
    0.000000     0.098811     0.000001     0.000000 


Riemann tensor components (sliced by upper index λ):
R^0_{μνρ} :
      0.000000   0.000000   0.000000  

## Riemann and Ricci Tensors

In differential geometry and general relativity, the **Riemann curvature tensor** and its contraction, the **Ricci tensor**, play a central role in describing the intrinsic curvature of spacetime.

---

### Christoffel Symbols

The **Christoffel symbols of the second kind** $ \Gamma^\alpha_{\mu\nu} $ are defined from the metric $ g_{\mu\nu} $ by:

$$
\Gamma^\alpha_{\mu\nu} = \frac{1}{2} g^{\alpha\lambda} \left( \partial_\mu g_{\lambda\nu} + \partial_\nu g_{\lambda\mu} - \partial_\lambda g_{\mu\nu} \right)
$$

They represent the coefficients of the Levi-Civita connection \( \nabla \), which is uniquely defined by two properties:
- **Metric compatibility**: $ \nabla g = 0 $
- **Zero torsion**: $ \nabla_X Y - \nabla_Y X = [X, Y] $

They appear in the **geodesic equation**:

$$
\frac{d^2 x^\alpha}{d\tau^2} + \Gamma^\alpha_{\mu\nu} \frac{dx^\mu}{d\tau} \frac{dx^\nu}{d\tau} = 0
$$

---

### Riemann Curvature Tensor

The **Riemann tensor** $ R^\alpha_{\beta\mu\nu} $ measures the failure of covariant derivatives to commute:

$$
R^\alpha_{\ \beta\mu\nu} = \partial_\mu \Gamma^\alpha_{\beta\nu} - \partial_\nu \Gamma^\alpha_{\beta\mu} + \Gamma^\alpha_{\mu\rho} \Gamma^\rho_{\beta\nu} - \Gamma^\alpha_{\nu\rho} \Gamma^\rho_{\beta\mu}
$$

Alternatively, for vector fields $ X, Y, Z $, it is defined geometrically by:

$$
R(X, Y)Z = \nabla_Y \nabla_X Z - \nabla_X \nabla_Y Z - \nabla_{[X, Y]} Z
$$

This tensor encodes how vectors change under parallel transport around infinitesimal loops, and thus fully characterizes local curvature.

---

### Ricci Tensor (Contraction)

The **Ricci tensor** is obtained by contracting the first and third indices of the Riemann tensor:

$$
R_{\mu\nu} = R^\lambda_{\ \mu\lambda\nu}
$$

It encodes the trace of curvature contributions from all directions and appears in the **Einstein field equations**:

$$
G_{\mu\nu} = R_{\mu\nu} - \frac{1}{2} R g_{\mu\nu}
$$

where $ R = g^{\mu\nu} R_{\mu\nu} $ is the scalar curvature.

---

### Summary

- $ \Gamma^\alpha_{\mu\nu} $ governs parallel transport via the Levi-Civita connection.
- $ R^\alpha_{\beta\mu\nu} $ measures curvature and torsion effects in spacetime.
- $ R_{\mu\nu} $ summarizes curvature into a rank-2 tensor and enters Einstein's equations.

These geometric objects are essential for formulating and solving the dynamics of spacetime in general relativity.


### Example: Interpreting $\Gamma^r_{tt}$ in the Kerr Metric

We consider the Kerr metric in Boyer–Lindquist coordinates $ (t, r, \vartheta, \phi) $, and focus on the Christoffel symbol:

$$
\Gamma^r_{tt} = \frac{c^2 r_s\, \Delta (r^2 - a^2 \cos^2 \vartheta)}{2 \Sigma^3}
$$

where the metric functions are:

- $ \Delta = r^2 - r_s r + a^2 $
- $ \Sigma = r^2 + a^2 \cos^2 \vartheta $

---

#### Meaning of this component

- $\Gamma^r_{tt}$ appears in the geodesic equation for the radial motion of a test particle:

$$
\frac{d^2 r}{d\tau^2} + \Gamma^r_{tt} \left( \frac{dt}{d\tau} \right)^2 + \cdots = 0
$$

- It encodes the influence of spacetime curvature on radial acceleration.
- The factor $(r^2 - a^2 \cos^2 \vartheta)$ reflects the deviation from spherical symmetry due to rotation (frame dragging).

---

#### Derivation

Christoffel symbols are computed from the metric:

$$
\Gamma^\lambda_{\mu\nu} = \frac{1}{2} g^{\lambda\sigma} \left( \partial_\mu g_{\nu\sigma} + \partial_\nu g_{\mu\sigma} - \partial_\sigma g_{\mu\nu} \right)
$$

We focus on the radial component:

$$
\Gamma^r_{tt} = \frac{1}{2} g^{rr} (-\partial_r g_{tt})
$$

From the Kerr metric:

$$
g_{tt} = -c^2 \left( 1 - \frac{r_s r}{\Sigma} \right), \quad g^{rr} = \frac{\Delta}{\Sigma}
$$

So we have:

$$
\Gamma^r_{tt} = \frac{1}{2} \cdot \frac{\Delta}{\Sigma} \cdot \left( \partial_r \left[ c^2 \left( 1 - \frac{r_s r}{\Sigma} \right) \right] \right)
$$

After simplification, this yields:

$$
\Gamma^r_{tt} = \frac{c^2 r_s\, \Delta (r^2 - a^2 \cos^2 \vartheta)}{2 \Sigma^3}
$$

---

#### Special cases

- **In Schwarzschild** ($ a = 0 $):

$$
\Gamma^r_{tt} = \frac{c^2 r_s (r - r_s)}{2 r^3}
$$

- **In Kerr**, the term $ a^2 \cos^2 \vartheta $ captures rotational effects and frame dragging.

---

This component is essential for computing the effective potential and stability of orbits in Kerr spacetime.



### Verifying the Christoffel Symbol $\Gamma^r_{tt}$ in the Kerr Metric

We consider the Kerr metric in Boyer–Lindquist coordinates and compute the Christoffel symbol:

$$
\Gamma^r_{tt} = \frac{r_s\, \Delta\, (r^2 - a^2 \cos^2 \theta)}{2 \Sigma^3}
$$

with:
- $ r_s = 2M $
- $ \Delta = r^2 - r_s r + a^2 $
- $ \Sigma = r^2 + a^2 \cos^2 \theta $

We evaluate this expression at the point: $X^\mu = (t,\ r,\ \theta,\ \phi) = (0,\ 10,\ \pi/2,\ 0)$

and Kerr parameters: $M = 1,\quad a = 0.935,\quad c = 1\ (\text{natural units})$

---

**Step-by-step computation:**

- $ r = 10,\quad \cos^2(\theta) = \cos^2(\pi/2) = 0 $
- $ \Sigma = r^2 + a^2 \cos^2 \theta = 100 $
- $ r_s = 2M = 2 $
- $ \Delta = r^2 - r_s r + a^2 = 100 - 20 + 0.874225 = 80.874225 $

We compute:

$$
\Gamma^r_{tt} = \frac{2 \cdot 80.874225 \cdot 100}{2 \cdot (100)^3}
= \frac{16174.845}{2000000} \approx 0.008087
$$

---

### Result

The symbolic computation gives: $\Gamma^r_{tt} \approx 0.008087$

Which exactly matches the numerical output:

```text
Γ^1_{μν} :
    0.008087     0.000000     0.000000    -0.007562 
    0.000000    -0.011284    -0.000000     0.000000 
    0.000000    -0.000000    -8.087422     0.000000 
   -0.007562     0.000000     0.000000    -8.080352 


### Verifying the Christoffel Symbol $\Gamma^t_{tr}$ in the Kerr Metric

We consider the Kerr metric in Boyer–Lindquist coordinates and compute the Christoffel symbol:

$$
\Gamma^t_{tr} = \frac{r_s\, (r^2 + a^2)(r^2 - a^2 \cos^2 \theta)}{2 \Sigma^2 \Delta}
$$

with:
- $r_s = 2M$
- $\Delta = r^2 - r_s r + a^2$
- $\Sigma = r^2 + a^2 \cos^2 \theta$

We evaluate this expression at the point:  
$X^\mu = (t,\ r,\ \theta,\ \phi) = (0,\ 10,\ \pi/2,\ 0)$

and Kerr parameters:  
$M = 1,\quad a = 0.935,\quad c = 1$ (natural units)

---

**Step-by-step computation:**

- $r = 10,\quad \cos^2(\theta) = \cos^2(\pi/2) = 0$
- $r_s = 2M = 2$
- $a^2 = 0.935^2 = 0.874225$
- $r^2 = 100$

Compute $\Sigma$, $\Delta$:
- $\Sigma = r^2 + a^2 \cos^2 \theta = 100$
- $\Delta = r^2 - r_s r + a^2 = 100 - 20 + 0.874225 = 80.874225$

Now compute numerator and denominator:

**Numerator:**

$$
r_s (r^2 + a^2)(r^2 - a^2 \cos^2 \theta) = 2 \cdot (100 + 0.874225) \cdot 100 = 2 \cdot 100.874225 \cdot 100 = 20174.845
$$

**Denominator:**

$$
2 \cdot \Sigma^2 \cdot \Delta = 2 \cdot 100^2 \cdot 80.874225 = 2 \cdot 10000 \cdot 80.874225 = 1617484.5
$$

---

### Result

$$
\Gamma^t_{tr} = \frac{20174.845}{1617484.5} \approx 0.012474
$$

Which exactly matches the numerical output:

```text
Γ^0_{μν} :
    0.000000     0.012474     0.000000     ...
    ...
